In [0]:
# from pyspark.sql.functions import *
# from pyspark.sql.types import *

# Bronze_path = "abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net/orders"

# # Read Bronze stream
# df = spark.readStream.format("delta").load(Bronze_path)

# # Correct schema (matching producer)
# schema = StructType([
#     StructField("order_id", StringType(), True),
#     StructField("customer_id", StringType(), True),
#     StructField("order_date", StringType(), True),
#     StructField("items", ArrayType(
#         StructType([
#             StructField("product_id", StringType(), True),
#             StructField("quantity", IntegerType(), True),
#             StructField("price", IntegerType(), True),
#             StructField("amount", IntegerType(), True)
#         ])
#     )),
#     StructField("total_amount", IntegerType(), True),
#     StructField("is_dirty", BooleanType(), True)
# ])

# # Parse JSON
# parsed_df = df.withColumn(
#     "data",
#     from_json(col("json_data"), schema)
# )

# # Flatten root
# flattened_df = parsed_df.select("data.*")

# # Explode items (IMPORTANT)
# silver_df = flattened_df.withColumn("item", explode("items")) \
#     .select(
#         "order_id",
#         "customer_id",
#         "order_date",
#         col("item.product_id").alias("product_id"),
#         col("item.quantity").alias("quantity"),
#         col("item.price").alias("price"),
#         col("item.amount").alias("item_amount"),
#         "total_amount",
#         "is_dirty"
#     )

# display(silver_df, checkpointLocation = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/checkpoints/silver_df")

In [0]:


# validated_df = flattened_df \
#     .withColumn("is_valid_customer", col("customer_id").isNotNull()) \
#     .withColumn("is_valid_amount", col("total_amount") > 0) \
#     .withColumn("is_valid_quantity", col("quantity") > 0) \
#     .withColumn("is_valid_price", col("price") > 0) \
#     .withColumn("is_valid_date", col("order_date").rlike("^\d{4}-\d{2}-\d{2}$")) \
#     .withColumn("is_not_dirty_flag", col("is_dirty") == False)


#     validated_df = validated_df.withColumn(
#     "quarantine_reason",
#     concat_ws(",",
#         when(~col("is_valid_customer"), "NULL_CUSTOMER_ID"),
#         when(~col("is_valid_amount"), "INVALID_TOTAL_AMOUNT"),
#         when(~col("is_valid_quantity"), "INVALID_QUANTITY"),
#         when(~col("is_valid_price"), "INVALID_PRICE"),
#         when(~col("is_valid_date"), "INVALID_DATE"),
#         when(~col("is_not_dirty_flag"), "DIRTY_FLAG_TRUE")
#     )
# )
    


# clean_df = validated_df.filter(
#     col("is_valid_customer") &
#     col("is_valid_amount") &
#     col("is_valid_quantity") &
#     col("is_valid_price") &
#     col("is_valid_date") &
#     col("is_not_dirty_flag")
# )

# quarantine_df = validated_df.filter(
#     ~(
#         col("is_valid_customer") &
#         col("is_valid_amount") &
#         col("is_valid_quantity") &
#         col("is_valid_price") &
#         col("is_valid_date") &
#         col("is_not_dirty_flag")
#     )
# )


# clean_df = clean_df.withColumn(
#     "order_date",
#     to_date(col("order_date"), "yyyy-MM-dd")
# ).withColumn(
#     "ingestion_time",
#     current_timestamp()
# ).drop(
#     "is_valid_customer",
#     "is_valid_amount",
#     "is_valid_quantity",
#     "is_valid_price",
#     "is_valid_date",
#     "is_not_dirty_flag",
#     "quarantine_reason"
# )

# quarantine_df = quarantine_df.withColumn(
#     "quarantine_time",
#     current_timestamp()
# )



In [0]:
# silver fact streaming notebook

from pyspark.sql.functions import *
from pyspark.sql.types import *

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
Bronze_path    = "abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net/orders"
Silver_path    = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/orders"
Quarantine_path= "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/quarantine/orders"
Control_path   = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/control/orders"
Checkpoint_silver     = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/order_checkpoints/silver"
Checkpoint_quarantine = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/checkpoints/quarantine"
Checkpoint_control    = "abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/checkpoints/control"

# ─────────────────────────────────────────
# STEP 1 — READ BRONZE STREAM
# ─────────────────────────────────────────
df = spark.readStream.format("delta").load(Bronze_path)

# ─────────────────────────────────────────
# STEP 2 — DEFINE SCHEMA & PARSE JSON
# ─────────────────────────────────────────
schema = StructType([
    StructField("order_id",      StringType(),  True),
    StructField("customer_id",   StringType(),  True),
    StructField("order_date",    StringType(),  True),
    StructField("items", ArrayType(
        StructType([
            StructField("product_id", StringType(),  True),
            StructField("quantity",   IntegerType(), True),
            StructField("price",      IntegerType(), True),
            StructField("amount",     IntegerType(), True)
        ])
    ), True),
    StructField("store_id", StringType(), True),
    StructField("region_id", StringType(), True),
    StructField("total_amount",  IntegerType(), True),
    StructField("is_dirty",      BooleanType(), True)
])

parsed_df = df.withColumn("data", from_json(col("json_data"), schema))
flattened_df = parsed_df.select("data.*")

# ─────────────────────────────────────────
# STEP 3 — EXPLODE ITEMS → silver_df
# ─────────────────────────────────────────
silver_df = (
    flattened_df
    .withColumn("item", explode("items"))
    .select(
        "order_id",
        "customer_id",
        "store_id",
        "region_id",
        "order_date",
        col("item.product_id").alias("product_id"),
        col("item.quantity").alias("quantity"),
        col("item.price").alias("price"),
        col("item.amount").alias("item_amount"),
        "total_amount",
        "is_dirty"
    )
)
# ─────────────────────────────────────────
# STEP 4 — VALIDATE  (runs on silver_df so
#           quantity / price columns exist)
# ─────────────────────────────────────────
validated_df = (
    silver_df
    .withColumn("is_valid_order_id", col("order_id").isNotNull())
    .withColumn("is_valid_store_id", col("store_id").isNotNull())
    .withColumn("is_valid_region_id", col("region_id").isNotNull())
    .withColumn("is_valid_product_id", col("product_id").isNotNull())
    .withColumn("is_valid_customer", col("customer_id").isNotNull())
    .withColumn("is_valid_amount",   col("total_amount") > 0)
    .withColumn("is_valid_quantity", col("quantity") > 0)
    .withColumn("is_valid_price",    col("price") > 0)
    .withColumn("is_valid_date",     col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
    .withColumn("is_not_dirty_flag", col("is_dirty") == False)
)

validated_df = validated_df.withColumn(
    "quarantine_reason",
    concat_ws(",",
        when(~col("is_valid_order_id"), "NULL_ORDER_ID"),
        when(~col("is_valid_store_id"), "NULL_STORE_ID"),
        when(~col("is_valid_region_id"),"NULL_REGION_ID"),
        when(~col("is_valid_product_id"),"NULL_PRODUCT_ID"),      
        when(~col("is_valid_customer"), "NULL_CUSTOMER_ID"),
        when(~col("is_valid_amount"),   "INVALID_TOTAL_AMOUNT"),
        when(~col("is_valid_quantity"), "INVALID_QUANTITY"),
        when(~col("is_valid_price"),    "INVALID_PRICE"),
        when(~col("is_valid_date"),     "INVALID_DATE"),
        when(~col("is_not_dirty_flag"), "DIRTY_FLAG_TRUE")
    )
)

# ─────────────────────────────────────────
# STEP 5 — SPLIT CLEAN vs QUARANTINE
# ─────────────────────────────────────────
all_valid = (
    col("is_valid_order_id") &
    col("is_valid_store_id") &
    col("is_valid_region_id") &
    col("is_valid_product_id") &
    col("is_valid_customer") &
    col("is_valid_amount")   &
    col("is_valid_quantity") &
    col("is_valid_price")    &
    col("is_valid_date")     &
    col("is_not_dirty_flag")
)

validation_cols = [
    "is_valid_order_id", "is_valid_store_id", "is_valid_region_id", "is_valid_product_id",
    "is_valid_customer", "is_valid_amount", "is_valid_quantity",
    "is_valid_price", "is_valid_date", "is_not_dirty_flag"
]

clean_df = (
    validated_df
    .filter(all_valid)
    .withColumn("order_date",     to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("ingestion_time", current_timestamp())
    .drop(*validation_cols, "quarantine_reason")
)

quarantine_df = (
    validated_df
    .filter(~all_valid)
    .withColumn("quarantine_time", current_timestamp())
)

# ─────────────────────────────────────────
# STEP 6 — CONTROL TABLE LOGIC
# Each micro-batch writes one summary row:
#   batch_id | run_time | total | clean | quarantine | duplicate
# ─────────────────────────────────────────
control_schema = StructType([
    StructField("batch_id",          LongType(),      False),
    StructField("run_time",          TimestampType(), False),
    StructField("source_layer",      StringType(),    False),
    StructField("target_layer",      StringType(),    False),
    StructField("total_records",     LongType(),      False),
    StructField("clean_records",     LongType(),      False),
    StructField("quarantine_records",LongType(),      False),
    StructField("duplicate_records", LongType(),      False),
    StructField("status",            StringType(),    False),
    StructField("error_message",     StringType(),    True)
])

def write_control_row(batch_df, batch_id, layer_src, layer_tgt):
    """
    Computes per-batch statistics from the validated DataFrame
    and appends one row to the Delta control table.
    """
    try:
        total       = batch_df.count()
        clean_count = batch_df.filter(all_valid).count()
        quar_count  = total - clean_count

        # Duplicate detection: same order_id + product_id appearing > 1 time
        dup_count = (
            batch_df
            .groupBy("order_id", "product_id")
            .count()
            .filter(col("count") > 1)
            .agg(spark_sum(col("count") - 1).alias("dups"))
            .collect()[0]["dups"] or 0
        )

        status = "SUCCESS" if total > 0 else "EMPTY_BATCH"

        control_row = spark.createDataFrame(
            [(batch_id,
              datetime.now(),
              layer_src,
              layer_tgt,
              total,
              clean_count,
              quar_count,
              dup_count,
              status,
              None)],
            schema=control_schema
        )

    except Exception as e:
        control_row = spark.createDataFrame(
            [(batch_id,
              datetime.now(),
              layer_src,
              layer_tgt,
              0, 0, 0, 0,
              "FAILED",
              str(e))],
            schema=control_schema
        )

    (
        control_row.write
        .format("delta")
        .mode("append")
        .save(Control_path)
    )


# ─────────────────────────────────────────
# STEP 7 — WRITE STREAMS
# foreachBatch lets us fan-out to multiple
# sinks + control table in one pass.
# ─────────────────────────────────────────
from datetime import datetime
from pyspark.sql.functions import sum as spark_sum

def process_batch(batch_df, batch_id):
    # Re-apply transformations (foreachBatch receives a static DF)
    val = (
        batch_df
        .withColumn("is_valid_order_id", col("order_id").isNotNull())
        .withColumn("is_valid_store_id", col("store_id").isNotNull())
        .withColumn("is_valid_region_id", col("region_id").isNotNull())
        .withColumn("is_valid_product_id", col("product_id").isNotNull())
        .withColumn("is_valid_customer", col("customer_id").isNotNull())
        .withColumn("is_valid_amount",   col("total_amount") > 0)
        .withColumn("is_valid_quantity", col("quantity") > 0)
        .withColumn("is_valid_price",    col("price") > 0)
        .withColumn("is_valid_date",     col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"))
        .withColumn("is_not_dirty_flag", col("is_dirty") == False)
        .withColumn(
            "quarantine_reason",
            concat_ws(",",
                when(~col("is_valid_order_id"), "NULL_ORDER_ID"),
                when(~col("is_valid_store_id"), "NULL_STORE_ID"),
                when(~col("is_valid_region_id"),"NULL_REGION_ID"),
                when(~col("is_valid_product_id"),"NULL_PRODUCT_ID"),      
                when(~col("is_valid_customer"), "NULL_CUSTOMER_ID"),
                when(~col("is_valid_amount"),   "INVALID_TOTAL_AMOUNT"),
                when(~col("is_valid_quantity"), "INVALID_QUANTITY"),
                when(~col("is_valid_price"),    "INVALID_PRICE"),
                when(~col("is_valid_date"),     "INVALID_DATE"),
                when(~col("is_not_dirty_flag"), "DIRTY_FLAG_TRUE")
            )
        )
    )

    clean = (
        val.filter(all_valid)
        .withColumn("order_date",     to_date(col("order_date"), "yyyy-MM-dd"))
        .withColumn("ingestion_time", current_timestamp())
        .drop(*validation_cols, "quarantine_reason")
    )

    quarantine = (
        val.filter(~all_valid)
        .withColumn("quarantine_time", current_timestamp())
    )

    # Write clean
    (clean.write
        .format("delta")
        .mode("append")
        .save(Silver_path))

    # Write quarantine
    (quarantine.write
        .format("delta")
        .mode("append")
        .save(Quarantine_path))

    # Write control table row
    write_control_row(val, batch_id, "bronze", "silver")


# Kick off the stream
query = (
    silver_df.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", Checkpoint_silver)
    .trigger(processingTime="1 minute")  # swap for .trigger(processingTime="1 minute") if continuous
    .start()
)

from datetime import datetime, timedelta
import time

end_time = datetime.now() + timedelta(hours=12)

while datetime.now() < end_time:
    if not query.isActive:
        break
    time.sleep(60)

query.stop()

query.awaitTermination()

In [0]:
required_cols = {"store_id", "region_id"}
missing_cols = required_cols - set(silver_df.columns)

if missing_cols:
    print(
        "clean_df cannot be displayed because upstream silver_df is missing required columns: "
        + ", ".join(sorted(missing_cols))
    )
else:
    display(
        clean_df,
        checkpointLocation="abfss://silver-layer@steusadlesgen0526.dfs.core.windows.net/orders/display_checkpoint"
    )